# Module 06: Normalizing Flows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/06-flow-models/notebook.ipynb)

**GPU recommended:** No (2D flows train in seconds on CPU).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 1. Change of Variables in 1D

A normalizing flow transforms a simple base distribution (e.g. a Gaussian) into a complex
target distribution through an invertible function. The change of variables formula tells us
how the density transforms:

$$p_X(x) = p_Z(f^{-1}(x)) \left| \det \frac{\partial f^{-1}}{\partial x} \right|$$

In 1D this simplifies to:

$$p_X(x) = p_Z(z) \left| \frac{dz}{dx} \right|$$

where $z = f^{-1}(x)$ and $x = f(z)$.

We demonstrate this with a simple invertible function $x = f(z) = z + \alpha \cdot \tanh(z)$.

In [ ]:
torch.manual_seed(42)

alpha = 0.8

def forward_transform(z):
    """x = f(z) = z + alpha * tanh(z). Invertible for |alpha| < 1."""
    return z + alpha * torch.tanh(z)

def forward_log_det_jacobian(z):
    """log |dx/dz| = log |1 + alpha * (1 - tanh^2(z))|"""
    return torch.log(torch.abs(1.0 + alpha * (1.0 - torch.tanh(z) ** 2)))

# Sample from base distribution (standard Gaussian)
num_samples = 50000
z_samples = torch.randn(num_samples)

# Transform through f
x_samples = forward_transform(z_samples)

# Compute the transformed density analytically on a grid
z_grid = torch.linspace(-4, 4, 1000)
x_grid = forward_transform(z_grid)

# p_Z(z) = standard normal density
log_pz = -0.5 * z_grid ** 2 - 0.5 * np.log(2 * np.pi)

# p_X(x) = p_Z(z) * |dz/dx| = p_Z(z) / |dx/dz|
log_px = log_pz - forward_log_det_jacobian(z_grid)

print(f"Base distribution: standard Gaussian")
print(f"Transform: x = z + {alpha} * tanh(z)")
print(f"z samples range: [{z_samples.min():.2f}, {z_samples.max():.2f}]")
print(f"x samples range: [{x_samples.min():.2f}, {x_samples.max():.2f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Base distribution
axes[0].hist(z_samples.numpy(), bins=100, density=True, alpha=0.7, color="steelblue")
axes[0].plot(z_grid.numpy(), torch.exp(log_pz).numpy(), "k-", linewidth=2, label="N(0,1) density")
axes[0].set_title("Base distribution p_Z(z)")
axes[0].set_xlabel("z")
axes[0].legend()

# The transform
axes[1].plot(z_grid.numpy(), x_grid.numpy(), "k-", linewidth=2)
axes[1].plot(z_grid.numpy(), z_grid.numpy(), "--", color="gray", linewidth=1, label="identity")
axes[1].set_title(f"Transform: x = z + {alpha} * tanh(z)")
axes[1].set_xlabel("z")
axes[1].set_ylabel("x")
axes[1].legend()

# Transformed distribution
axes[2].hist(x_samples.numpy(), bins=100, density=True, alpha=0.7, color="coral")
# Sort x_grid for clean line plot
sort_idx = x_grid.argsort()
axes[2].plot(x_grid[sort_idx].numpy(), torch.exp(log_px)[sort_idx].numpy(), "k-", linewidth=2,
             label="Analytical p_X(x)")
axes[2].set_title("Transformed distribution p_X(x)")
axes[2].set_xlabel("x")
axes[2].legend()

plt.tight_layout()
plt.show()

print("The histogram of transformed samples matches the analytical density,")
print("verifying the change of variables formula.")

---
## 2. Affine Coupling Layer (RealNVP Style)

The RealNVP architecture uses **affine coupling layers** to build invertible transforms
in higher dimensions. The key idea: split the input into two halves. One half passes
through unchanged, while the other is affinely transformed using parameters computed
from the first half.

For a 2D input $z = [z_1, z_2]$:
- $x_1 = z_1$ (identity)
- $x_2 = z_2 \cdot \exp(s(z_1)) + t(z_1)$ (affine transform)

where $s$ and $t$ are neural networks. The Jacobian is triangular, so the
log-determinant is simply $s(z_1)$.

### Generate 2D target data

We create a two-moons dataset as our target distribution.

In [ ]:
def make_moons(num_samples, noise=0.05):
    """Generate two-moons dataset."""
    torch.manual_seed(42)
    n = num_samples // 2
    theta_top = torch.linspace(0, np.pi, n)
    theta_bottom = torch.linspace(0, np.pi, n)

    x_top = torch.stack([torch.cos(theta_top), torch.sin(theta_top)], dim=1)
    x_bottom = torch.stack([1 - torch.cos(theta_bottom), 1 - torch.sin(theta_bottom) - 0.5], dim=1)

    data = torch.cat([x_top, x_bottom], dim=0)
    data += noise * torch.randn_like(data)

    # Shuffle
    perm = torch.randperm(data.shape[0])
    return data[perm]

target_data = make_moons(2000, noise=0.05)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(target_data[:, 0].numpy(), target_data[:, 1].numpy(), s=3, alpha=0.5, color="coral")
ax.set_title("Target distribution: two moons")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Define the affine coupling layer

In [ ]:
class AffineCouplingLayer(nn.Module):
    """RealNVP-style affine coupling layer for 2D data.

    Split input into two halves. One passes through unchanged,
    the other is affinely transformed using parameters from the first.
    """
    def __init__(self, mask, hidden_dim=64):
        super().__init__()
        self.mask = mask  # binary mask: which dims pass through unchanged

        # Networks that compute scale (s) and translation (t)
        self.scale_net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Tanh(),  # bound scale for stability
        )
        self.translate_net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z):
        """Forward: z -> x. Returns x and log_det_jacobian."""
        z_masked = z * self.mask               # unchanged part
        z_change = z * (1 - self.mask)          # part to transform

        # Compute affine params from the masked (unchanged) part
        s = self.scale_net(z_masked[:, self.mask.bool()])
        t = self.translate_net(z_masked[:, self.mask.bool()])

        # Apply affine transform to the other part
        z_change_input = z[:, (~self.mask.bool())].unsqueeze(1)
        x_change = z_change_input * torch.exp(s) + t

        # Assemble output
        x = z.clone()
        x[:, (~self.mask.bool())] = x_change.squeeze(1)

        log_det = s.sum(dim=-1)
        return x, log_det

    def inverse(self, x):
        """Inverse: x -> z."""
        x_masked = x * self.mask

        s = self.scale_net(x_masked[:, self.mask.bool()])
        t = self.translate_net(x_masked[:, self.mask.bool()])

        x_change = x[:, (~self.mask.bool())].unsqueeze(1)
        z_change = (x_change - t) * torch.exp(-s)

        z = x.clone()
        z[:, (~self.mask.bool())] = z_change.squeeze(1)
        return z

# Quick test: verify invertibility
torch.manual_seed(42)
mask = torch.tensor([1.0, 0.0])  # first dim unchanged
layer = AffineCouplingLayer(mask)
test_z = torch.randn(5, 2)
test_x, test_logdet = layer(test_z)
test_z_recovered = layer.inverse(test_x)

print(f"Invertibility error: {(test_z - test_z_recovered).abs().max().item():.2e}")
print(f"Log-det shape: {test_logdet.shape}")

---
## 3. Stacking Coupling Layers

A single coupling layer leaves one half of the dimensions unchanged. By alternating
which half is masked, we allow all dimensions to be transformed. Stacking multiple
layers increases the expressiveness of the flow.

In [ ]:
class RealNVP2D(nn.Module):
    """Stack of alternating affine coupling layers for 2D data."""
    def __init__(self, num_layers=6, hidden_dim=64):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            # Alternate masks: [1,0], [0,1], [1,0], ...
            if i % 2 == 0:
                mask = torch.tensor([1.0, 0.0])
            else:
                mask = torch.tensor([0.0, 1.0])
            self.layers.append(AffineCouplingLayer(mask, hidden_dim))

    def forward(self, z):
        """Forward: base -> data. Returns x and total log_det."""
        total_log_det = torch.zeros(z.shape[0])
        x = z
        intermediates = [z.detach().clone()]
        for layer in self.layers:
            x, log_det = layer(x)
            total_log_det = total_log_det + log_det.squeeze(-1)
            intermediates.append(x.detach().clone())
        return x, total_log_det, intermediates

    def inverse(self, x):
        """Inverse: data -> base."""
        z = x
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z

    def log_prob(self, x):
        """Compute log p(x) using change of variables."""
        z = self.inverse(x)
        # Compute forward log_det from z
        _, total_log_det, _ = self.forward(z)
        # Base distribution log prob
        log_pz = -0.5 * (z ** 2).sum(dim=-1) - np.log(2 * np.pi)
        return log_pz + total_log_det

torch.manual_seed(42)
flow = RealNVP2D(num_layers=6, hidden_dim=64)
num_params = sum(p.numel() for p in flow.parameters())
print(f"RealNVP2D: {len(flow.layers)} coupling layers, {num_params} parameters")

---
## 4. Training with Exact Log-Likelihood

A key advantage of normalizing flows over VAEs or GANs: we can compute the exact
log-likelihood of the data. Training maximizes $\log p(x)$ directly, which is
equivalent to minimizing the negative log-likelihood (NLL).

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log p(x_i) = -\frac{1}{N} \sum_{i=1}^{N} \left[ \log p_Z(f^{-1}(x_i)) + \log \left| \det \frac{\partial f^{-1}}{\partial x} \right| \right]$$

In [ ]:
torch.manual_seed(42)
flow = RealNVP2D(num_layers=6, hidden_dim=64)
optimizer = torch.optim.Adam(flow.parameters(), lr=1e-3)

target_data = make_moons(2000, noise=0.05)
num_epochs = 1000
batch_size = 256
loss_history = []

for epoch in range(num_epochs):
    # Random mini-batch
    idx = torch.randint(0, target_data.shape[0], (batch_size,))
    batch = target_data[idx]

    # Compute NLL loss
    z = flow.inverse(batch)
    _, log_det, _ = flow.forward(z)
    log_pz = -0.5 * (z ** 2).sum(dim=-1) - np.log(2 * np.pi)
    log_px = log_pz + log_det.squeeze()
    nll_loss = -log_px.mean()

    optimizer.zero_grad()
    nll_loss.backward()
    optimizer.step()

    loss_history.append(nll_loss.item())

    if epoch % 200 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch:5d} | NLL: {nll_loss.item():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, alpha=0.3, color="steelblue")
# Smoothed curve
window = 50
if len(loss_history) > window:
    smoothed = np.convolve(loss_history, np.ones(window) / window, mode="valid")
    ax.plot(range(window - 1, len(loss_history)), smoothed, color="darkblue", linewidth=2,
            label=f"Smoothed (window={window})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Negative Log-Likelihood")
ax.set_title("Training Loss: Exact NLL")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Sampling via Inverse Flow

To generate new samples, we sample $z \sim \mathcal{N}(0, I)$ from the base distribution
and pass it through the forward flow $x = f(z)$. Because the flow is invertible and
trained to map the base to the data distribution, the output should resemble the
training data.

In [ ]:
torch.manual_seed(42)
num_gen = 2000

# Sample from base and push through the flow
z_new = torch.randn(num_gen, 2)
with torch.no_grad():
    x_gen, _, _ = flow.forward(z_new)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(z_new[:, 0].numpy(), z_new[:, 1].numpy(), s=3, alpha=0.3, color="steelblue")
axes[0].set_title("Base distribution z ~ N(0, I)")
axes[0].set_aspect("equal")
axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)

axes[1].scatter(target_data[:, 0].numpy(), target_data[:, 1].numpy(), s=3, alpha=0.3, color="coral")
axes[1].set_title("Training data (two moons)")
axes[1].set_aspect("equal")

axes[2].scatter(x_gen[:, 0].numpy(), x_gen[:, 1].numpy(), s=3, alpha=0.3, color="mediumseagreen")
axes[2].set_title("Generated samples (flow)")
axes[2].set_aspect("equal")

plt.tight_layout()
plt.show()

### Visualizing the step-by-step transformation

We can observe how each coupling layer progressively warps the Gaussian
into the target shape.

In [ ]:
torch.manual_seed(42)
z_vis = torch.randn(1500, 2)
with torch.no_grad():
    _, _, intermediates = flow.forward(z_vis)

num_steps = len(intermediates)
fig, axes = plt.subplots(1, num_steps, figsize=(3 * num_steps, 3))

for i, (ax, points) in enumerate(zip(axes, intermediates)):
    ax.scatter(points[:, 0].numpy(), points[:, 1].numpy(), s=1, alpha=0.3, color="steelblue")
    if i == 0:
        ax.set_title("z (base)")
    elif i == num_steps - 1:
        ax.set_title(f"x (output)")
    else:
        ax.set_title(f"Layer {i}")
    ax.set_aspect("equal")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-3, 3)

plt.suptitle("Distribution at each coupling layer", y=1.02)
plt.tight_layout()
plt.show()

### Density evaluation on a grid

Unlike GANs or basic VAEs, normalizing flows provide exact density evaluation.
We evaluate log p(x) on a fine grid to visualize the learned density.

In [ ]:
grid_size = 200
x_range = torch.linspace(-1.5, 2.5, grid_size)
y_range = torch.linspace(-1.0, 1.5, grid_size)
xx, yy = torch.meshgrid(x_range, y_range, indexing="ij")
grid_points = torch.stack([xx.flatten(), yy.flatten()], dim=1)

with torch.no_grad():
    log_probs = flow.log_prob(grid_points)
    probs = torch.exp(log_probs).reshape(grid_size, grid_size)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].contourf(xx.numpy(), yy.numpy(), probs.numpy(), levels=30, cmap="viridis")
axes[0].set_title("Learned density p(x)")
axes[0].set_aspect("equal")
plt.colorbar(im, ax=axes[0])

axes[1].contourf(xx.numpy(), yy.numpy(), probs.numpy(), levels=30, cmap="viridis", alpha=0.5)
axes[1].scatter(target_data[:, 0].numpy(), target_data[:, 1].numpy(), s=2, alpha=0.3, color="coral",
                label="Training data")
axes[1].set_title("Density + training data overlay")
axes[1].set_aspect("equal")
axes[1].legend(markerscale=5)

plt.tight_layout()
plt.show()

---
## 6. Flow Matching (Comparison)

Flow matching is a recent alternative to normalizing flows. Instead of learning an
invertible mapping with tractable Jacobians, flow matching learns a **velocity field**
$v_\theta(x, t)$ that defines an ODE transporting the base distribution to the data
distribution over a time interval $t \in [0, 1]$.

The training objective is simple: given a data point $x_1$ and a noise sample $x_0 \sim \mathcal{N}(0,I)$,
define the interpolation $x_t = (1-t) x_0 + t x_1$. The target velocity is $v^* = x_1 - x_0$.
We train the network to predict this velocity:

$$\mathcal{L} = \mathbb{E}_{t, x_0, x_1} \| v_\theta(x_t, t) - (x_1 - x_0) \|^2$$

At inference, we solve the ODE $\frac{dx}{dt} = v_\theta(x, t)$ from $t=0$ to $t=1$
using Euler steps.

In [ ]:
class VelocityNetwork(nn.Module):
    """MLP that predicts velocity given position and time."""
    def __init__(self, data_dim=2, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden_dim),  # +1 for time
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x, t):
        """x: (batch, data_dim), t: (batch, 1)"""
        return self.net(torch.cat([x, t], dim=-1))

torch.manual_seed(42)
velocity_net = VelocityNetwork(data_dim=2, hidden_dim=128)
fm_optimizer = torch.optim.Adam(velocity_net.parameters(), lr=1e-3)

fm_losses = []
num_fm_epochs = 2000
batch_size = 256

for epoch in range(num_fm_epochs):
    # Sample data and noise
    idx = torch.randint(0, target_data.shape[0], (batch_size,))
    x1 = target_data[idx]                         # data samples
    x0 = torch.randn(batch_size, 2)               # noise samples
    t = torch.rand(batch_size, 1)                  # random time

    # Interpolate
    xt = (1 - t) * x0 + t * x1

    # Target velocity
    target_velocity = x1 - x0

    # Predict velocity
    pred_velocity = velocity_net(xt, t)

    loss = F.mse_loss(pred_velocity, target_velocity)

    fm_optimizer.zero_grad()
    loss.backward()
    fm_optimizer.step()

    fm_losses.append(loss.item())

    if epoch % 400 == 0 or epoch == num_fm_epochs - 1:
        print(f"Epoch {epoch:5d} | Flow Matching Loss: {loss.item():.4f}")

In [ ]:
def euler_sample(velocity_net, num_samples, num_steps=100):
    """Generate samples by solving the ODE with Euler method."""
    dt = 1.0 / num_steps
    x = torch.randn(num_samples, 2)
    trajectory = [x.clone()]

    with torch.no_grad():
        for step in range(num_steps):
            t = torch.full((num_samples, 1), step * dt)
            v = velocity_net(x, t)
            x = x + v * dt
            if step % (num_steps // 5) == 0 or step == num_steps - 1:
                trajectory.append(x.clone())

    return x, trajectory

torch.manual_seed(42)
fm_samples, fm_trajectory = euler_sample(velocity_net, 2000, num_steps=100)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(target_data[:, 0].numpy(), target_data[:, 1].numpy(), s=3, alpha=0.3, color="coral")
axes[0].set_title("Training data")
axes[0].set_aspect("equal")

with torch.no_grad():
    z_nvp = torch.randn(2000, 2)
    x_nvp, _, _ = flow.forward(z_nvp)
axes[1].scatter(x_nvp[:, 0].numpy(), x_nvp[:, 1].numpy(), s=3, alpha=0.3, color="mediumseagreen")
axes[1].set_title("RealNVP samples")
axes[1].set_aspect("equal")

axes[2].scatter(fm_samples[:, 0].numpy(), fm_samples[:, 1].numpy(), s=3, alpha=0.3, color="mediumpurple")
axes[2].set_title("Flow Matching samples")
axes[2].set_aspect("equal")

plt.suptitle("Normalizing Flow vs Flow Matching")
plt.tight_layout()
plt.show()

### Visualize the ODE trajectory

Flow matching defines a continuous-time transport. We can visualize how samples
move from noise to data over time.

In [ ]:
num_traj = len(fm_trajectory)
fig, axes = plt.subplots(1, num_traj, figsize=(3 * num_traj, 3))

for i, (ax, pts) in enumerate(zip(axes, fm_trajectory)):
    ax.scatter(pts[:, 0].numpy(), pts[:, 1].numpy(), s=1, alpha=0.3, color="mediumpurple")
    t_val = i / max(num_traj - 1, 1)
    ax.set_title(f"t = {t_val:.2f}")
    ax.set_aspect("equal")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-3, 3)

plt.suptitle("Flow Matching: ODE trajectory from noise to data", y=1.02)
plt.tight_layout()
plt.show()

### Comparison: velocity field visualization

In [ ]:
# Visualize the learned velocity field at several time steps
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
time_steps = [0.0, 0.25, 0.5, 0.75]

grid_n = 20
gx = torch.linspace(-3, 3, grid_n)
gy = torch.linspace(-3, 3, grid_n)
gxx, gyy = torch.meshgrid(gx, gy, indexing="ij")
grid_pts = torch.stack([gxx.flatten(), gyy.flatten()], dim=1)

for ax, t_val in zip(axes, time_steps):
    t_tensor = torch.full((grid_pts.shape[0], 1), t_val)
    with torch.no_grad():
        velocities = velocity_net(grid_pts, t_tensor)

    ax.quiver(
        grid_pts[:, 0].numpy(), grid_pts[:, 1].numpy(),
        velocities[:, 0].numpy(), velocities[:, 1].numpy(),
        alpha=0.7, scale=30
    )
    ax.set_title(f"Velocity field at t={t_val}")
    ax.set_aspect("equal")
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)

plt.suptitle("Learned velocity field v(x, t)")
plt.tight_layout()
plt.show()

---
## Summary

This notebook covered normalizing flows and flow matching:

1. **Change of variables** -- the mathematical foundation: transforming densities through invertible functions
2. **Affine coupling layers** -- the RealNVP building block: split, transform, and preserve invertibility
3. **Stacked flows** -- alternating masks let all dimensions be transformed, with step-by-step visualization
4. **Exact log-likelihood** -- a key advantage of normalizing flows: train by directly maximizing data likelihood
5. **Sampling** -- generate new data by pushing base samples through the learned flow
6. **Flow matching** -- a modern alternative that learns a velocity field and generates via ODE integration

Key trade-offs:
- **Normalizing flows**: exact likelihood, exact invertibility, but require specialized architectures (coupling layers)
- **Flow matching**: simpler training objective, works with any network architecture, but no exact likelihood and requires ODE solving at inference